Tasks:
1. Test the state preparation
2. Insert the state preparation circuit
3. Use the P function in the implementation

# One-Dimensional Fermi-Hubbard Model

## Overview

## Introduction
The Fermi-Hubbard model is a fundamental corner step of condensed matter physics, describing the interplay between kinetic energy and repulsion interaction between electrons in a solid. Although very simple, in certain parameter regimes, the model showcases a variety strongly correlated phenomena, such as the metal-insulator transition (Mott physics), antiferromagnetism, inhomogeneous phases, and unconventional Fermi liquids.

In order to motivate or conceptually derive the model, consider of a metalic material composed of atoms organized in a regular pattern, i.e., a lattice. We assume the temperature is sufficiently low, so the that the vibrations of the atoms can be neglected and the atoms are essentially held in a fix position in the  lattice sites. Moreover, typically each atom has a handful of relevant energy levels, however, in order to simplify the problem we will consider only a single energy level at each lattice site (one energy level per atom). In the  metal, valance electrons move freely between the metal atoms, leading to the expected conductive behaviour. The movement of the electrons between the sites is dictated by the overlap of the spatial wave function on different sites. Since the atomic wave-functions decay exponentially with the distance to the nuclei, we consider only the dominant contribution, giving rise to hopping of electrons only between adjacent lattice sites.
These consideration lead to the first the hopping term, constituting the kinetic energy contribution: $-J \sum_{\langle i,j \rangle, \sigma} \left ( c_{i\sigma}^\dagger c_{j\sigma} + c_{i\sigma}^\dagger c_{j\sigma}\right)$, where $c_{i\sigma}$ is the fermionic annihilation operator on the $i$'th lattice site and $\sigma =
\uparrow, \downarrow$ spin. In addition, $\langle i,j\rangle$ designates that the sum is only over nearest-neighbors.

A second contribution arises from the repulsive Coulomb interaction between two electrons. Such interaction is strongest between electrons on the same atom. We consider only this dominant contribution, adding an energy penalty of $U$, for two-electrons on the same site. In addition, Pauli's exclusion principle dictates that two electrons cannot be in the same quantum state, therefore the repulsive interaction can only be between electrons with different spins: $U \sum_{j} c_{j \uparrow}^\dagger c_{j \uparrow}  c_{j\downarrow}^\dagger c_{j\downarrow}$.

Overall, the Fermi-Hubbard Hamiltonian is given by: $$H_{HF} = -J \sum_{\langle i,j \rangle, \sigma} \left ( c_{i\sigma}^\dagger c_{j\sigma} + c_{i\sigma}^\dagger c_{j\sigma}\right) + U \sum_{j} n_{j \uparrow}  n_{j\downarrow}~~, \tag{1}$$ where $ n_{j \sigma}= c_{j \sigma}^\dagger c_{j \sigma}$ is the number operator.


The model constitutes a key tool in the investigation of transition metal oxides, organic conductors  and cuprates, which exhibit high-temperature superconductivity and exotic quantum magnetism properties and unconventional symmetry breaking.
Despite its apparent simplicity, the model is notoriously changing, in 1D the Fermi-Hubbard model is solvable via Bethe ansatz [[2](#LiebWu)], while the 2D and 3D cases have not exact solution (for general hopping and interaction coefficients, $J$ and $U$). However, in 3D quantum phenomena are suppressed and the model typically exhibit classical behaviour, well described by mean-field theories.
Numerical studies are also limited beyond specific doping regimes (doping modifies the fraction of electrons/lattice sites). Perturbation theory becomes invalid when strong correlations between the electron emerge, while Monte-Carlo methods are restricted by the sign problem away from half filling (at half filling the number of electrons equals the number lattice sites).

## Present Notebook

The present notebook focuses on a digital simulation of the 1D Fermi-Hubbard (FH) model, following the experimental demonstration by Google AI Quantum [[1](#google_paper)].
We demonstrate how the model is mapped to a qubit system, allowing a quantum computer to simulate non-trivial dynamics of a strongly correlated fermionic system. The dynamics exhibit spin-charge separation, where spin and charge excitations propagate at different velocities, in the non-equilibrium regime.
The digital nature of the simulation allows flexability and control over the initiail states, measured observables, and also enables time-reversal.
The physics of the model are conveniently analysed by measuring the evolution of the spin densities $$\rho_j^{\pm}(t) = \langle n_{j,\uparrow}\rangle \pm \langle n_{j,\downarrow} \rangle~~,$$ and spin  spread $\kappa^{\pm}(t)=\sum_{j=1}^L |j-(L+1)/2|\rho^{\pm}_{j}(t)$
Crucially, the experimental realization in [[1](#google_paper)] showcased that such dynamics are accessible on the present Noisy Intermediate-Scale Quantum (NISQ) devices.


## Classiq Implementation

We implement the Hamiltonian simulation in three steps, utilizing Classiq's built-in functions.

1. Initial state-preparation: Efficiently prepare the ground state of a non-interacting FH system in the presence of an external spin dependent potential. The ground state is a Gaussian Fermionic state, creating a localized density peak, acting as a wavepacket source.
2. Time-evolution: Abruptly turn on the two-electron repulsion interaction and turn off the external potential, leading to dynamics governed by Hamiltonian (1). Utilizing a first-order Trotter expansion, the system is evolved in time.
3. Measurement: The spin densities are evaluated.

Repetition of these three steps many times produces the evolution of expectation values of spin-density in time, $\{\rho^{\pm}_{j}(t)\}$.

We consider a simplified model consisting of a periodic lattice of $L=4$ sites, and study the dynamics in a varying range of interaction strengths, $U/J\in [0,5]$.

We begin by importing the required software packages and defining global constants

In [778]:
# Uncomment to install openfermion python package
#!pip install openfermion

In [779]:
import numpy as np
from openfermion.linalg.givens_rotations import givens_decomposition

from classiq import *
from classiq.qmod.quantum_function import QFunc

In [780]:
L = 4  # number of lattice sites
M = 2 * L  # total number of fermionic nodes
N = 2  # number of electrons
NUM_ITERS = 10
J = 1  # hopping strength
tau = 0.4 / t  # Trotter step time interval
T = NUM_ITERS * tau

### Initial State Preparation

The initial state is chosen to be the ground state of the non-interacting (i.e, quadratic in the fermionic creation/annihilation operators) Hamiltonian $$H_{0} =-J \sum_{j, \sigma} \left ( c_{j,\sigma}^\dagger c_{j+1,\sigma} + c_{j+1,\sigma}^\dagger c_{j,\sigma}\right) + \sum_{j,\sigma}\epsilon_{j,\sigma}n_{j,\sigma} ~~,$$ where the spin-up fermions feel a Gaussian attractive potential $$\epsilon_{j,\uparrow} = -\lambda \exp \left[ -\frac{(j-m)^2}{2 w^2}\right]~~,$$ where spin-down fermions feel no potential, $\epsilon_{j,\downarrow}$. The parameters $\lambda$, $m$ and $w$ dictate the precise shape and strength of the potential.
Such potential creates a localized density peak of spin-up fermions and a flatter distribution for the spin-down fermions. As a result, the simulation begins with localized charge and spin densities. This state is generally not an eigenstate of the interacting Hamiltonian, constituting a highly excited (non-equilibrium) of the interacting system.

As all ground states of non-interacting fermionic Hamiltonian, the ground state is a fermionic Gaussian state, which can be prepared in a worst-case circuit depth of $O(L^2)$ [[2](#fermionic_gaussian_state)].


### Preparation of a  Slater determinant
For a general number conserving quadratic fermionic Hamiltonian $H = \sum_{\mu\nu}c_\mu^\dagger h_{\mu\nu}c_\nu$, where $h$ is a Hermitian (single particle) matrix, and $\mu = (i,\sigma)$ (similarly $\nu$) designates both lattice site and spin quantum numbers.
By diagonalizing the single particle Hamiltonian $h= Q^\dagger D Q$, where $Q$ is unitary and $D = \text{diag}(\epsilon_1,...,\epsilon_L)$, we obtain $H=\sum_k \epsilon_k d_k^\dagger d_k$, where $$d_k = \sum_i Q_{ki}c_i~~. \tag{4}$$
For a fixed particle number $N$ the ground state is given by
$$|\psi_{g.s}\rangle =\Pi_{k=1}^N {\cal U}c_k^\dagger |0^n\rangle =  \Pi_{k=1}^N d_k^\dagger |0^n\rangle ~~,$$
where ${\cal U} c_j^\dagger {\cal U}^\dagger = d_j $.
Alternatively, the state can be expressed as a Slater determinant, or in a Gaussian form: $\exp\left ( \sum_{\mu,\nu}c_\mu^\dagger A_{\mu,\nu}c_\nu\right )| \text{ref}\rangle$ for a anti-Hermitian matrix $A$ and a reference state with $N$ occupied nodes (the fermionic Gaussian state can also describe cases where the number of electrons are not fixed).
The diagonalization of $h$ scales only polynomially with the number of lattice sites, $O(L^3)$ and can be efficiently done on a classical computer.

In order to prepare the desired ground state we focus on a part of $Q$ which corresponds to the occupied nodes, and denote the  $N$ by $M=2L$ matrix describing these modes by $\tilde{Q}$
The transformation (4) corresponds to a single particle basis transformation, which can be decomposed into a product of elementary two-mode rotations, called Given rotations. Each rotation can be expressed as
$$
\begin{pmatrix}
\mathcal{G}c_k^\dagger\mathcal{G}\\
\mathcal{G}c_j^\dagger\mathcal{G}
\end{pmatrix}
=
G(\theta,\varphi)\,
\begin{pmatrix}
c_k\\
c_j
\end{pmatrix}~~,
 $$
and the form of a Given rotation is  $$G_{jk}(\theta,\varphi) = \begin{pmatrix}
\cos(\theta) & -e^{i\varphi}\sin(\theta)\\
\sin(\theta) & e^{i\varphi}\cos(\theta)
\end{pmatrix}~~. $$
The basis transformation can be expressed in terms of the single-particle transformation $U$: The decomposition of $U$ to Given rotations is obtained by a modified QR decomposition. We utilize the invariance of the Slater determinant (up to a global phase) under the mapping $\tilde{Q}\rightarrow V\tilde{Q}$, where $V$ is a unitary transformation. For the transformation of basis to be valid we require that the first $N$ rows of $V\tilde{Q}$ are equal to $U$, or alternatively $V\tilde{Q}U^\dagger = (I_N, \boldsymbol{0})$. Each Given transformation operates only on two columns and is determined so to nullify elements in the upper right part of the matrix, see Technical Notes section for further details. Due to the orthogonality of the rows of $\tilde{Q}$, some transformations nullify more than a single matrix element. As a result, the total number of required Given transformations are $m = N(M-N)$, where $N$ is the number of electrons and $M$ is the number of fermionic modes. The diagonalization procedure results in a product of Given rotations $$U = G_m\cdots G_2 G_1~~.$$

After the Jordan-Wigner transformation, each two-mode ($j,k$) rotations correspond to a rotation in the single particle subspace of the two qubits $j$ and $k$ ($|01\rangle$ and $|10\rangle$). This completely, defines the state preparation circuit in terms of a sequence two-qubit rotations.
The gate complexity is $O(m) = O(N^2)$ (worst case achieved for $M=N/2$), and parallelization leads to a circuit depth of $M-1$.

We first define a function mapping lattice and spin degrees of freedom to qubit number.

In [781]:
def qubit_idx(site: int, spin: int):
    """
    Maps lattice site and spin to qubit indices.

    Args:
        site (int): Lattice site index, the range [0,L-1]
        spin (int): Spin index, either 0 or 1
    """
    return 2 * (site % L) + (spin % 2)

The Given rotations are evaluated utilizing `OpenFermion` function `givens_decomposition`, producing a circuit description consisting of a list of tuples, where each tuple contains Given rotations which can be performed simultaneously. Each Given rotation is characterized by a tuple $(i,j, \theta, \varphi)$, where $i$ and $j$ are the columns the rotation transforms and $\theta$ and $\varphi$ are the rotations angles.

In [782]:
@qfunc
def G(theta: float, phi: float, qba: QArray[QBit, 2]):
    U = [
        [1, 0, 0, 0],
        [0, np.cos(theta), -np.exp(1j * phi) * np.sin(theta), 0],
        [0, np.sin(theta), np.exp(1j * phi) * np.cos(theta), 0],
        [0, 0, 0, 1],
    ]
    unitary(U, qba)


@qfunc
def given_rotation(gate: list[int, int, float, float], qba: QArray[QBit, M]) -> None:
    i, j, theta, phi = gate
    G(theta, phi, [qba[i], qba[j]])


@qfunc
def prepare_slater_det(h: list[list[float, M], M], N: int, qba: QArray[QBit, M]):
    """
    Prepares the ground state associated with the single electron matrix h.
    The Hamiltonian satisfies H = \sum_{\mu,\nu}c_{\mu}^\dagger h_{\mu,\nu} c_{\nu}

    Args:
        h (ndarray): single electron matrix
        N (int): number of electrons
        qba (list[QBit]): list of qubits
    """
    # diagonalizing h
    D, Q = np.linalg.eigh(h)
    Q_tilde = Q[:N, :]  # basis transformation matrix
    decomposition, _, _ = givens_decomposition(Q_tilde)
    circuit_description = list(reversed(decomposition))
    for layer in circuit_description:
        for gate in layer:
            given_rotation(list(gate), qba)

### Model Variables

In [783]:
def sym(A):
    """
    Symmetrizes the matrix A
    """
    return (A + A.T) / 2


def kinetic_energy(L: int, J: float) -> np.ndarray:
    """
    Builds single electron matrix of nearest-neighbor hopping term, hopping strength t, associated with an L site periodic lattice.
    """
    K = np.zeros((2 * L, 2 * L), dtype=float)
    for site in range(L):
        for spin in range(2):
            mu = qubit_idx(site, spin)
            nu = qubit_idx(site + 1, spin)
            K[mu, nu] = -J
    K = 2 * sym(K)
    return K


def spin_potential(L: int, spin=0, parameters: tuple[float] = (1, 1, 1)) -> np.ndarray:
    """
    Builds single electron , matrix associated with the external potential. Associated with an L site periodic lattice.
    """
    lam, mean, std = parameters
    V = np.zeros((2 * L, 2 * L), dtype=float)
    for site in range(L):
        mu = qubit_idx(site, spin)
        V[mu, mu] = -lam * np.exp(-((mu - mean) ** 2) / (2 * std**2))
    return V


def prepare_single_electron_hamiltonian(
    L: int, J: float, parameters: tuple
) -> np.ndarray:
    return kinetic_energy(L, J) + spin_potential(L, spin=0, parameters=parameters)

Prepare the single electron matrix $h$:

In [784]:
lam = 0.1
mean = L / 2
std = L / 6
h = prepare_single_electron_hamiltonian(L, J, parameters=(lam, mean, std))

In [785]:
@qfunc
def main(qba: Output[QArray[QBit, M]]) -> None:
    allocate(M, qba)
    prepare_slater_det(h, N, qba)


qprog = synthesize(main)

The state preparation quantum circuit. The Given rotation, operating on two qubits, is decomposed into the basis gates.


ADD the CIRCUIT...

DVerifying the preparation of the ground state

In [786]:
# classical calculation
Dmat, Vmat = np.linalg.eigh(h)
# single electron test


# assert(np.isclose(classical_ground_state == result))

### Time-Evolution

The propagation stage approximates the time-evolution operator $U = \exp(-i H t)$, where
$$ H =-J \sum_{j, \sigma} \left ( c_{j,\sigma}^\dagger c_{j+1,\sigma} + c_{j,\sigma}^\dagger c_{j+1,\sigma}\right) + U \sum_{j} n_{j \uparrow}  n_{j\downarrow}  \tag{3}$$ is the 1D version of Eq. (1), where periodic conditions are assumed, $j\equiv j \mod L$.

The propagation of the initial state with respect to Eq. (3), simulates an effective quench of the system at initial time, abruptly changing the Hamiltonian $H_0\rightarrow H$. As a consequence, the initial state is a highly excited state of $H$.

In order to minimize the circuit depth of the quantum circuit it is beneficial to decompose the Hamiltonian into four sets of terms, where all the operators in a set commute with one another. The terms correspond to an even and odd edge hopping terms and even and odd site interaction terms:
$$H = H_{\text{hop-even}} +H_{\text{hop-odd}} +H_{\text{int-even}}+ H_{\text{int-odd}} ~~.$$
The odd hopping Hamiltonian term includes the hopping terms between $1\leftrightarrow
2$ and $3\leftrightarrow
4$ neighbouring sites ($j$ is odd in Eq. (3)), while the odd hopping term includes the odd edge pairs ($2\leftrightarrow
3$, $4\leftrightarrow
1$, even $j$).

The evolution operator $U$ is approximated by a first-order Trotter expansion, where each Trotter step consists of five stages.
$$U = e^{-i H t} \approx \left(e^{-i H \tau}\right)^{t/\tau}~~, $$ with $$e^{-i H \tau}  \approx e^{-i H_{\text{hop-even}}\tau} e^{-i H_{\text{int-even}}\tau} e^{-i H_{\text{int-odd}}\tau} e^{-i H_{\text{hop-odd}}\tau}. $$
Each component in the Trotter step can be achieved by simultaneous application (circuit depth of one) of simple two-qubit gates.

An additional algorithmic trick simplifying the circuit and reducing the circuit depth even further. Between the even and odd interaction terms a fermionic mode swap operation is conducted,  effectively performing a cyclic shifting of the fermionic modes. Crucially, this operation swaps the logical ordering of the qubits, while preserving the fermionic parity sign (using an iSWAP gates). Alternatively, the operation modifies the mapping between physical qubits and logical fermionic. As a result, the Hamiltonian terms are interpreted relative to the new ordering, effectively swapping the odd and even edge.
The incorporation of the swap mode operation, allows parallelism, locality (only nearest neighbor gates, bypassing the need for long Jordan Wigner strings).
After the application of $\exp(-i H_{\text{hop-even}}\tau)$ an inverse fermionic mode swap is needed to swap the even and odd edges back, allowing application of the consecutive Trotter step. Since the even edge hopping term commutes with the inverse fermionic mod swap, these operations are merged to a single stage.

Overall, the time-evolution involves iteration (each iteration corresponds to a single Trotter step) of a five-step procedure:
1. Hopping operation on odd edges.
2. Interaction on odd sites
3. Fermionic mode swap
4. Interaction on even sites
5. Hopping operation on even sites + inverse fermionic mode swap




Setting the Hamiltonian parameters

In [787]:
U = 0.1 * J  # J is set to unity in the beginning of the notebook

We begin by defining utility functions

In [788]:
## Rotation in the |01>, |10> subspace


# K(theta) = exp(-i*(theta/2)*(XX + YY))
@qfunc
def K(theta: float, target: QArray[QBit, 2]) -> None:
    Unitary = [
        [1, 0, 0, 0],
        [0, np.cos(theta), -1j * np.sin(theta), 0],
        [0, -1j * np.sin(theta), np.cos(theta), 0],
        [0, 0, 0, 1],
    ]
    unitary(Unitary, target)


# P(phi) = exp(-i*(phi/2)*(I-Z_{\mu}-Z_{\nu}+Z_{\mu}Z_{\nu}))
@qfunc
def P(phi: float, target: QArray[QBit, 2]) -> None:
    RZ(-phi, target[0])
    RZ(-phi, target[1])
    RZZ(phi, target)

To propagate the system state we transform the fermion Hamilotonian to a qubit representation utilizing the Jordan-Wigner transformation.

A single hopping term transforms as
$$c_{\mu}^{\dagger} c_{\nu} + c_{\mu}^{\dagger} c_{\nu}  \xrightarrow{\text{JW}} \frac{1}{2} \left( X_{\mu}X_{\nu} + Y_{\mu}Y_{\nu} \right)~~,$$
while an interaction term gives $$n_{\mu}n_{\nu}  \xrightarrow{\text{JW}} = \frac{1}{4}\left(I -Z_\mu -Z_\nu + Z_\mu Z_\nu \right)~~. $$

As a consequence, the five stages can be performed by implementation of the basic unitaries $K(\theta)= \exp\left(-i \frac{\theta}{2}\left(XX + YY \right)\right)$ and $P(\phi) = \exp\left(-i \frac{\phi}{2}\left(I-Z_{\mu}-Z_{\nu}+Z_{\mu}Z_{\nu} \right)\right)$. The functions `K` and `P` above implement the corresponding unitary transformations.

The first stage, including odd hopping terms, is obtained by simultaneous application of $K(\theta = -\tau J)$ on different paris of qubits. Similarly, the second and fourth stages, involving the odd and even interactions are achieved with simultaneous application of $P(\phi = \tau U/2)$ on the corresponding pairs of qubits. The third stage, constituting the fermionic mode swap, is implemented by simultaneous iSWAP gates which is equivalent to $K(\theta = -\pi/2)$. The last stage includes a merge of even-edge hopping operation, and an inverse fermionic mode swap operation. Naturally, since these operation commute, the transformation is obtained by $K(\theta = -\tau J + \pi/2)$ two-qubit gates.

We implement the simultaneous operations on the qubit array utilizing Classiq's built-in `repeat` function within the `hop` and `interact` functions.

In [789]:
def hop(pairs: list[tuple[int, int]], theta: float, qba: QArray[QBit, M]) -> None:
    for spin in {0, 1}:
        for pair in pairs:
            site1, site2 = pair
            K(
                theta,
                [
                    qba[qubit_idx(site=site1, spin=spin)],
                    qba[qubit_idx(site=site2, spin=spin)],
                ],
            )


def interact(sites: list[int], phi, qba: QArray[QBit, M]) -> None:
    for spin in {0, 1}:
        for site in sites:
            RZ(-phi, qba[qubit_idx(site=site, spin=spin)])
            RZ(-phi, qba[qubit_idx(site=site, spin=spin + 1)])
            RZZ(
                phi,
                [
                    qba[qubit_idx(site=site, spin=spin)],
                    qba[qubit_idx(site=site, spin=spin + 1)],
                ],
            )
            # P(phi,
            #    [qba[qubit_idx(site=site, spin=spin)],
            #    qba[qubit_idx(site=site, spin=spin+1)]]
            # )

Defining the odd and even pairs and sites

In [790]:
ODD_PAIRS = [(i, (i + 1) % L) for i in range(0, L, 2)]
EVEN_PAIRS = [((i + 1) % L, i) for i in range(0, L, 2)]
ODD_SITES = list(range(0, L, 2))
EVEN_SITES = list(range(1, L, 2))

Gathering all the stages together to form the Trotter step

In [791]:
def trotter_step(tau: float, J: float, U: float, qba: QArray[QBit, L]) -> None:

    # stage 1 - hopping on odd-edge pairs
    hop(pairs=ODD_PAIRS, theta=-tau * J, qba=qba)

    # stage 2 - interaction on odd site
    interact(sites=ODD_SITES, phi=tau * U / 2, qba=qba)

    # stage 3 - fermionic mode swap
    hop(pairs=EVEN_PAIRS, theta=-np.pi / 2, qba=qba)

    # stage 4 - interaction on even sites
    interact(sites=EVEN_SITES, phi=tau * U / 2, qba=qba)

    # stage 5 - hopping on even-edge pairs and inverse fermionic mode swap
    hop(pairs=EVEN_PAIRS, theta=-tau * J + np.pi / 2, qba=qba)

### Propagation and Execution

In [792]:
@qfunc
def main(qba: Output[QArray[QBit, M]]) -> None:
    allocate(M, qba)
    prepare_slater_det(h, N, qba)

    for _ in range(NUM_ITERS):
        trotter_step(tau, J, U, qba)


qprog = synthesize(main)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/397SVsR5o5MzxTdQ7C3doKLZybq


#### Measurement

The physics of the model are analyzed by measuring the charge and spin densities, $\rho_j^{\pm} = \langle n_{j,\uparrow}\rangle \pm \langle n_{j,\downarrow} \rangle$. The charge density, $\rho_j^{+}$, corresponds to the average number of electrons on the $j$'th site, while the spin density, $\rho_j^{-}$, is the net spin on the same site. We also measure the spin spreads $\kappa^{\pm}(t)$, defined above.

The number operators map to local Pauli operators $n_{\mu} = c^\dagger_{\mu}c_{\mu} = (1-Z_\mu)/2$, where $\mu=(j,\sigma)$ encodes a site, spin pair.


In [795]:
@qfunc
def charge_density(site: int, M: int) -> SparsePauliOp:
    s, I = SparsePauliOp([], M)
    for spin in {0, 1}:
        idx = qubit_idx(site=site, spin=spin)
        s += (Pauli.I(idx) - Pauli.Z(idx)) / 2
    return s


@qfunc
def spin_density(site: int, M: int) -> SparsePauliOp:
    s, I = SparsePauliOp([], M)
    idx_up, idx_down = qubit_idx(site=site, spin=0), qubit_idx(site=site, spin=1)
    s_up = (Pauli.I(idx_up) - Pauli.Z(idx_up)) / 2
    s_down = (Pauli.I(idx_down) - Pauli.Z(idx_down)) / 2
    return s_up - s_down


def spread(density: list, L: int) -> float:
    s = 0
    for site in range(L):
        s += np.abs(site - (L + 1) / 2) * density[site]
        return s

In [801]:
# Execute
s = SparsePauliOp([], M)
s += Pauli.X(0)
with ExecutionSession(qprog) as es:
    res = es.estimate(s)
    # charge_density_result = es.estimate(charge_density(1))

In [807]:
print(res.value)

(-0.0107421875+0j)


## Technical Notes

### Given Rotations
TODO

## References

<a id='google_paper'>[1] </a>: Arute, F., Arya, K., Babbush, R., Bacon, D., Bardin, J. C., Barends, R., ... & Zanker, S. (2020). Observation of separated dynamics of charge and spin in the Fermi-Hubbard model. [arXiv:2010.07965](https://arxiv.org/abs/2010.07965).

<a id='fermionic_gaussian_state'>[2] Jiang, Z., Sung, K. J., Kechedzhi, K., Smelyanskiy, V. N., & Boixo, S. (2018). Quantum algorithms to simulate many-body physics of correlated fermions. [Physical Review Applied, 9(4), 044036](https://arxiv.org/abs/1711.05395).

**Googl's paper** NOTE: SPIN CHARGE SEPARATION IS KNOWN IN THE LOW ENERGY LUTTINGER LIQUID THEORY, THEY SHOWCASE THIS ALSO IN THE HIGHLY EXCITED QUENCH, OUTSIDE THE STRICT VALIDITY OF THE LUTTINGER THEORY.

They consider a 8 lattice sites, each one can have 2 spins = 16 qubits, interaction strengths $U/J$ of 0 to around 5. Mainly quarter filling ($N_\uparrow = N_\downarrow = 2$)
Charge density spreads faster than spin density, the separation increases with interaction strength $U$. They quantify the spread using the measure $\kappa_\pm (t) = \sum_j |j-(L+1)/2|\rho_j^{\pm}(t)$, taking the time-derivative gives the effective propagation velocities.
The dynamics start from a localized density peak, constituting a highly excited state, yet
spin-charge separation still emerges.

The trotter step is done if 5-stages: hopping term on odd sites, interaction on odd sites, fermionic mode swap, effectively performing a cyclic shift of the fermionic modes (swapping the logical ordering of the qubits, preserving the fermionic parity sign (uses an iSWAP gate)) What happens is that the mapping between physical qubits and logical fermionic modes changes, the Hamiltonian terms are interperted relative to the new ordering. Then interaction on the odd strings, and hopping term.

The essence of the fermionic shift is that it allows parallelism, locality (operations on nearest neighbor gates (no long Jordan Wigner strings)), reduces trotter step complexity, minimal depth. After the step they swap back by applying an inverse fermionic permutation of stage 3. Since the even-bond hopping gates and the inverse fermionic swaps commute with each other they merge the even hopping and the inverse swap.

- State preparation algorithm, preparing a non-interacting fermionic ground state with a localized density (The use deterministic fermionic Gaussian state preparation (two-mode rotations corresponding to orthogonal transformations on the fermionic modes), equivalent to preparing a Slater determinant. Depth is $O(L)-O(L^2)$)
- Time-evolution using the 5-stage procedure (for each Trotter step)
- Measurement